# Phase 5 — Anatomical Consistency Module (URI-CADS)

**Rescoped from the original plan**: the original idea (capsule/medulla/pelvis internal-compartment consistency) needs annotations this dataset doesn't have. What URI-CADS *does* give us -- a kidney mask and lesion boxes -- supports a different, still-genuine anatomical constraint:

> **Lesions are anatomically inside the kidney.** A predicted lesion region that falls outside the predicted/true kidney boundary is anatomically impossible.

This adds a **consistency loss** term that penalizes lesion-mask probability mass predicted *outside* the kidney region, on top of Phase 4's architecture. Everything else (heads, other losses) is identical to Phase 4 -- this notebook only adds one new loss term so the effect of the constraint is isolated and measurable.

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
import os

DRIVE_PROJECT_ROOT = '/content/drive/MyDrive/URI-CADS dataset'
DRIVE_ZIP_PATH = f'{DRIVE_PROJECT_ROOT}/uri_cads_archive.zip'
DRIVE_CHECKPOINTS_PATH = f'{DRIVE_PROJECT_ROOT}/checkpoints'
MANIFEST_SAVE_PATH = f'{DRIVE_PROJECT_ROOT}/manifest_with_splits.pkl'

LOCAL_ZIP_PATH = '/content/uri_cads_archive.zip'
LOCAL_DATA_DIR = '/content/uri_cads_data'
LOCAL_DATASET_DIR = os.path.join(LOCAL_DATA_DIR, 'bbdd_kidney')

os.makedirs(DRIVE_CHECKPOINTS_PATH, exist_ok=True)
print('Zip exists on Drive:', os.path.exists(DRIVE_ZIP_PATH))
print('Saved manifest exists:', os.path.exists(MANIFEST_SAVE_PATH))

Zip exists on Drive: True
Saved manifest exists: True


In [6]:
import shutil, zipfile, time

os.makedirs(LOCAL_DATA_DIR, exist_ok=True)
t0 = time.time()
shutil.copy(DRIVE_ZIP_PATH, LOCAL_ZIP_PATH)
with zipfile.ZipFile(LOCAL_ZIP_PATH, 'r') as zf:
    zf.extractall(LOCAL_DATA_DIR)
print(f'Copied + unzipped in {time.time()-t0:.1f}s')
print('Contents:', os.listdir(LOCAL_DATASET_DIR))

Copied + unzipped in 4.2s
Contents: ['images', 'masks', 'labels']


In [7]:
import pandas as pd

manifest = pd.read_pickle(MANIFEST_SAVE_PATH)
print(manifest['split'].value_counts())

split
train    1389
val       298
test      298
Name: count, dtype: int64


In [8]:
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU -- switch Runtime > Change runtime type > GPU before continuing')

True Tesla T4


## Lesion parsing + Dataset (identical to Phase 4)

In [9]:
import numpy as np

NUM_LOCAL_PATHOLOGY_IDS = 9


def parse_local_lesions(extra_lines):
    lesions = []
    for line in extra_lines:
        if not line.startswith('Bounding box'):
            continue
        _, values = line.split(':', 1)
        nums = [int(v) for v in values.split()]
        if len(nums) != 5:
            continue
        x, y, w, h, pathology_id = nums
        lesions.append({'bbox': (x, y, x + w, y + h), 'pathology_id': pathology_id})
    return lesions


def bbox_to_coarse_mask(bbox, height, width):
    mask = np.zeros((height, width), dtype=np.uint8)
    x_min, y_min, x_max, y_max = [int(round(v)) for v in bbox]
    mask[y_min:y_max, x_min:x_max] = 1
    return mask

In [10]:
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import scipy.io

IMG_SIZE = (512, 512)


def load_kidney_mask(mat_path):
    data = scipy.io.loadmat(mat_path)
    mask = data['mask']
    return np.asarray(mask).astype(np.uint8)


class URICADSMultiTaskDataset(Dataset):
    def __init__(self, manifest_df, img_size=IMG_SIZE):
        self.manifest = manifest_df.reset_index(drop=True)
        self.img_size = img_size

    def __len__(self):
        return len(self.manifest)

    def __getitem__(self, idx):
        row = self.manifest.iloc[idx]
        target_h, target_w = self.img_size

        image = Image.open(row['image_path']).convert('RGB')
        orig_width, orig_height = image.size
        image = image.resize((target_w, target_h), Image.BILINEAR)
        image = np.array(image).astype(np.float32) / 255.0
        image = (image - 0.5) / 0.5

        kidney_mask = load_kidney_mask(row['mask_path'])
        kidney_mask = np.array(Image.fromarray(kidney_mask).resize((target_w, target_h), Image.NEAREST))

        global_diagnosis = int(row['global_diagnosis'])
        global_pathologies = torch.tensor(row['global_pathologies'], dtype=torch.float32)

        scale_x = target_w / orig_width
        scale_y = target_h / orig_height
        lesions = parse_local_lesions(row['extra_lines'])

        lesion_union_mask = np.zeros((target_h, target_w), dtype=np.uint8)
        local_pathology_vec = np.zeros(NUM_LOCAL_PATHOLOGY_IDS, dtype=np.float32)
        for lesion in lesions:
            x_min, y_min, x_max, y_max = lesion['bbox']
            scaled_box = (x_min * scale_x, y_min * scale_y, x_max * scale_x, y_max * scale_y)
            box_mask = bbox_to_coarse_mask(scaled_box, target_h, target_w)
            lesion_union_mask = np.maximum(lesion_union_mask, box_mask)
            pid = lesion['pathology_id']
            if 1 <= pid <= NUM_LOCAL_PATHOLOGY_IDS:
                local_pathology_vec[pid - 1] = 1.0

        image_tensor = torch.from_numpy(image).permute(2, 0, 1).float()
        kidney_mask_tensor = torch.from_numpy(kidney_mask).float()
        lesion_mask_tensor = torch.from_numpy(lesion_union_mask).float()

        return {
            'image': image_tensor,
            'kidney_mask': kidney_mask_tensor,
            'global_diagnosis': torch.tensor(global_diagnosis, dtype=torch.float32),
            'global_pathologies': global_pathologies,
            'lesion_mask': lesion_mask_tensor,
            'local_pathologies': torch.from_numpy(local_pathology_vec),
            'case_id': row['case_id'],
        }


train_ds = URICADSMultiTaskDataset(manifest[manifest['split'] == 'train'])
val_ds = URICADSMultiTaskDataset(manifest[manifest['split'] == 'val'])
test_ds = URICADSMultiTaskDataset(manifest[manifest['split'] == 'test'])

print(f'Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}')

Train: 1389 | Val: 298 | Test: 298


## Model (identical architecture to Phase 4 -- the constraint is added purely in the loss function)

In [11]:
import torch.nn as nn
import torchvision

NUM_GLOBAL_PATHOLOGIES = len(manifest.iloc[0]['global_pathologies'])


class MultiTaskNetV2(nn.Module):
    def __init__(self, num_global_pathologies=2, num_local_pathologies=NUM_LOCAL_PATHOLOGY_IDS, pretrained=True):
        super().__init__()
        backbone = torchvision.models.resnet34(weights='IMAGENET1K_V1' if pretrained else None)
        self.encoder = nn.Sequential(*list(backbone.children())[:-2])

        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.diagnosis_head = nn.Linear(512, 1)
        self.global_pathology_head = nn.Linear(512, num_global_pathologies)
        self.local_pathology_head = nn.Linear(512, num_local_pathologies)

        def up_block(in_c, out_c):
            return nn.Sequential(
                nn.Conv2d(in_c, out_c, 3, padding=1),
                nn.BatchNorm2d(out_c),
                nn.ReLU(inplace=True),
                nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            )

        self.kidney_decoder = nn.Sequential(
            up_block(512, 256), up_block(256, 128), up_block(128, 64),
            up_block(64, 32), up_block(32, 16),
            nn.Conv2d(16, 1, kernel_size=1),
        )
        self.lesion_decoder = nn.Sequential(
            up_block(512, 256), up_block(256, 128), up_block(128, 64),
            up_block(64, 32), up_block(32, 16),
            nn.Conv2d(16, 1, kernel_size=1),
        )

    def forward(self, x):
        feat = self.encoder(x)
        pooled = self.global_pool(feat).flatten(1)

        diagnosis_logits = self.diagnosis_head(pooled).squeeze(1)
        global_pathology_logits = self.global_pathology_head(pooled)
        local_pathology_logits = self.local_pathology_head(pooled)

        kidney_seg_logits = self.kidney_decoder(feat).squeeze(1)
        lesion_seg_logits = self.lesion_decoder(feat).squeeze(1)

        return {
            'diagnosis': diagnosis_logits,
            'global_pathology': global_pathology_logits,
            'local_pathology': local_pathology_logits,
            'kidney_seg': kidney_seg_logits,
            'lesion_seg': lesion_seg_logits,
        }


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = MultiTaskNetV2(num_global_pathologies=NUM_GLOBAL_PATHOLOGIES).to(device)

Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /root/.cache/torch/hub/checkpoints/resnet34-b627a593.pth


100%|██████████| 83.3M/83.3M [00:00<00:00, 136MB/s]


## Loss -- Phase 4's 5 terms + the new anatomical consistency term

**Consistency loss**: penalizes predicted lesion probability that falls *outside* the ground-truth kidney mask. Uses the true kidney mask (not the model's own prediction) as the anatomical prior, since it's more stable during training than constraining against a still-learning kidney head.

`L_consistency = mean( lesion_pred_prob * (1 - kidney_mask_target) )`

This is `0` when all predicted lesion probability mass lies inside the true kidney region, and grows as the model predicts lesions outside the kidney boundary.

In [12]:
def dice_loss(pred_logits, target, eps=1e-6):
    pred = torch.sigmoid(pred_logits)
    pred_flat = pred.flatten(1)
    target_flat = target.flatten(1)
    intersection = (pred_flat * target_flat).sum(dim=1)
    union = pred_flat.sum(dim=1) + target_flat.sum(dim=1)
    dice = (2 * intersection + eps) / (union + eps)
    return 1 - dice.mean()


def anatomical_consistency_loss(lesion_logits, kidney_mask_target):
    """Penalizes lesion probability mass predicted outside the true kidney region."""
    lesion_prob = torch.sigmoid(lesion_logits)
    outside_kidney = 1.0 - kidney_mask_target
    violation = lesion_prob * outside_kidney
    return violation.mean()


bce_logits = nn.BCEWithLogitsLoss()


def compute_loss(outputs, batch):
    kidney_target = batch['kidney_mask'].to(device)

    loss_diagnosis = bce_logits(outputs['diagnosis'], batch['global_diagnosis'].to(device))
    loss_global = bce_logits(outputs['global_pathology'], batch['global_pathologies'].to(device))
    loss_local = bce_logits(outputs['local_pathology'], batch['local_pathologies'].to(device))
    loss_kidney_seg = dice_loss(outputs['kidney_seg'], kidney_target)
    loss_lesion_seg = dice_loss(outputs['lesion_seg'], batch['lesion_mask'].to(device))
    loss_consistency = anatomical_consistency_loss(outputs['lesion_seg'], kidney_target)

    # FIXED, EQUAL sum across all 6 terms -- Phase 6 optimizes these weights, not here.
    total_loss = (loss_diagnosis + loss_global + loss_local
                  + loss_kidney_seg + loss_lesion_seg + loss_consistency)

    breakdown = {
        'diagnosis': loss_diagnosis.item(),
        'global_pathology': loss_global.item(),
        'local_pathology': loss_local.item(),
        'kidney_seg': loss_kidney_seg.item(),
        'lesion_seg': loss_lesion_seg.item(),
        'consistency': loss_consistency.item(),
    }
    return total_loss, breakdown

## Training setup

In [13]:
BATCH_SIZE = 4
NUM_EPOCHS = 15
LEARNING_RATE = 1e-4

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

In [14]:
from tqdm.auto import tqdm


def dice_score(pred_logits, target, eps=1e-6):
    pred = (torch.sigmoid(pred_logits) > 0.5).float()
    pred_flat = pred.flatten(1)
    target_flat = target.flatten(1)
    intersection = (pred_flat * target_flat).sum(dim=1)
    union = pred_flat.sum(dim=1) + target_flat.sum(dim=1)
    return ((2 * intersection + eps) / (union + eps)).mean().item()


def consistency_violation_rate(lesion_logits, kidney_mask_target, eps=1e-6):
    """Fraction of predicted lesion probability mass that falls outside the kidney -- the metric
    version of the loss, for reporting how well the constraint is actually being satisfied."""
    lesion_prob = torch.sigmoid(lesion_logits)
    total_mass = lesion_prob.flatten(1).sum(dim=1) + eps
    outside_mass = (lesion_prob * (1.0 - kidney_mask_target)).flatten(1).sum(dim=1)
    return (outside_mass / total_mass).mean().item()


def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss = 0.0
    total_kidney_dice = 0.0
    total_lesion_dice = 0.0
    total_violation = 0.0
    n_batches = 0
    context = torch.enable_grad() if train else torch.no_grad()
    with context:
        for batch in tqdm(loader, desc='train' if train else 'val'):
            images = batch['image'].to(device)
            outputs = model(images)
            loss, _ = compute_loss(outputs, batch)

            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            kidney_target = batch['kidney_mask'].to(device)
            total_loss += loss.item()
            total_kidney_dice += dice_score(outputs['kidney_seg'], kidney_target)
            total_lesion_dice += dice_score(outputs['lesion_seg'], batch['lesion_mask'].to(device))
            total_violation += consistency_violation_rate(outputs['lesion_seg'], kidney_target)
            n_batches += 1

    return (total_loss / n_batches, total_kidney_dice / n_batches,
            total_lesion_dice / n_batches, total_violation / n_batches)

## Tiny sanity test -- run BEFORE the full loop

In [15]:
tiny_train = torch.utils.data.Subset(train_ds, range(20))
tiny_loader = DataLoader(tiny_train, batch_size=4, num_workers=0)
t0 = time.time()
run_epoch(tiny_loader, train=True)
print(f'20 samples took {time.time()-t0:.1f}s')

train:   0%|          | 0/5 [00:00<?, ?it/s]

20 samples took 2.8s


## Full training loop

In [16]:
best_val_score = 0.0
for epoch in range(1, NUM_EPOCHS + 1):
    t0 = time.time()
    train_loss, train_kidney_dice, train_lesion_dice, train_violation = run_epoch(train_loader, train=True)
    val_loss, val_kidney_dice, val_lesion_dice, val_violation = run_epoch(val_loader, train=False)
    elapsed = time.time() - t0

    val_score = (val_kidney_dice + val_lesion_dice) / 2

    print(f'Epoch {epoch:2d}/{NUM_EPOCHS} | '
          f'val_loss={val_loss:.4f} val_kidney_dice={val_kidney_dice:.4f} '
          f'val_lesion_dice={val_lesion_dice:.4f} val_violation_rate={val_violation:.4f} | {elapsed:.1f}s')

    if val_score > best_val_score:
        best_val_score = val_score
        checkpoint_path = os.path.join(DRIVE_CHECKPOINTS_PATH, 'phase5_anatomical_best.pt')
        torch.save(model.state_dict(), checkpoint_path)
        print(f'  -> new best combined dice={val_score:.4f}, checkpoint saved to Drive')

train:   0%|          | 0/348 [00:00<?, ?it/s]

val:   0%|          | 0/75 [00:00<?, ?it/s]

Epoch  1/15 | val_loss=3.2740 val_kidney_dice=0.6363 val_lesion_dice=0.2797 val_violation_rate=0.8489 | 81.1s
  -> new best combined dice=0.4580, checkpoint saved to Drive


train:   0%|          | 0/348 [00:00<?, ?it/s]

val:   0%|          | 0/75 [00:00<?, ?it/s]

...

val:   0%|          | 0/75 [00:00<?, ?it/s]

Epoch 14/15 | val_loss=2.1245 val_kidney_dice=0.8771 val_lesion_dice=0.3438 val_violation_rate=0.3516 | 83.5s


train:   0%|          | 0/348 [00:00<?, ?it/s]

val:   0%|          | 0/75 [00:00<?, ?it/s]

Epoch 15/15 | val_loss=2.1984 val_kidney_dice=0.8863 val_lesion_dice=0.3554 val_violation_rate=0.3582 | 82.8s
  -> new best combined dice=0.6208, checkpoint saved to Drive


## Test-set evaluation -- compare directly against Phase 4 (no consistency constraint)

In [17]:
from sklearn.metrics import roc_auc_score

test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

model.load_state_dict(torch.load(os.path.join(DRIVE_CHECKPOINTS_PATH, 'phase5_anatomical_best.pt')))
model.eval()

all_diag_probs, all_diag_targets = [], []
all_kidney_dice, all_lesion_dice, all_violation = [], [], []

with torch.no_grad():
    for batch in test_loader:
        images = batch['image'].to(device)
        outputs = model(images)
        kidney_target = batch['kidney_mask'].to(device)

        all_diag_probs.extend(torch.sigmoid(outputs['diagnosis']).cpu().numpy())
        all_diag_targets.extend(batch['global_diagnosis'].numpy())
        all_kidney_dice.append(dice_score(outputs['kidney_seg'], kidney_target))
        all_lesion_dice.append(dice_score(outputs['lesion_seg'], batch['lesion_mask'].to(device)))
        all_violation.append(consistency_violation_rate(outputs['lesion_seg'], kidney_target))

test_diag_auc = roc_auc_score(all_diag_targets, all_diag_probs)

print('PHASE 5 RESULTS (multi-task + anatomical consistency constraint):')
print(f'  Healthy/Pathological AUC     : {test_diag_auc:.4f}   (compare to Phase 4)')
print(f'  Kidney Segmentation Dice     : {np.mean(all_kidney_dice):.4f}   (compare to Phase 4)')
print(f'  Lesion (coarse) Seg. Dice    : {np.mean(all_lesion_dice):.4f}   (compare to Phase 4)')
print(f'  Consistency Violation Rate   : {np.mean(all_violation):.4f}   (lower = better; this is the metric the constraint targets)')
print()
print('The key comparison for the paper: does the violation rate drop vs. Phase 4 (no constraint),')
print('and does lesion Dice hold steady or improve -- that is the evidence the anatomical')
print('consistency module is doing something real, not just adding loss-function noise.')

PHASE 5 RESULTS (multi-task + anatomical consistency constraint):
  Healthy/Pathological AUC     : 0.8758   (compare to Phase 4)
  Kidney Segmentation Dice     : 0.8970   (compare to Phase 4)
  Lesion (coarse) Seg. Dice    : 0.3425   (compare to Phase 4)
  Consistency Violation Rate   : 0.3790   (lower = better; this is the metric the constraint targets)

The key comparison for the paper: does the violation rate drop vs. Phase 4 (no constraint),
and does lesion Dice hold steady or improve -- that is the evidence the anatomical
consistency module is doing something real, not just adding loss-function noise.
